# Steps 3-4: baseline and hyperparameter sweep (MC_Maze)
Run cells top to bottom (Shift+Enter). Each section ends with a plot.

**Task (co-smoothing):** the model sees 137 "held-in" neurons and must predict the spikes of
45 "held-out" neurons it never sees. **co-bps** = bits/spike gained over predicting each
neuron's average rate (0 = no better than average, higher = better).

In [1]:
import json, os, time
import numpy as np, matplotlib.pyplot as plt
from neuralfm.data import load_mc_maze
from neuralfm.baselines import smooth, mean_rate
from neuralfm.metrics import co_bps, bootstrap_ci, poisson_ll
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

d = load_mc_maze(os.environ.get("NEURALFM_DATA", "../data/000128/sub-Jenkins"),
                 cache="../results/mc_maze_5ms.npz")
for k, v in d.items():
    print(f"{k:10s} {v.shape}  (trials, time bins of 5 ms, neurons)")

train_in   (1721, 140, 137)  (trials, time bins of 5 ms, neurons)
train_out  (1721, 140, 45)  (trials, time bins of 5 ms, neurons)
val_in     (573, 140, 137)  (trials, time bins of 5 ms, neurons)
val_out    (573, 140, 45)  (trials, time bins of 5 ms, neurons)


## 3a. Look at the data: spike raster for one trial

In [ ]:
trial = 0
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for a, key, title in [(ax[0], "train_in", "Held-in neurons (model input)"),
                      (ax[1], "train_out", "Held-out neurons (to predict)")]:
    t_idx, n_idx = np.nonzero(d[key][trial])
    a.scatter(t_idx * 5, n_idx, s=1.5, c="k")
    a.set(title=title, xlabel="time (ms)", ylabel="neuron")
plt.tight_layout(); plt.show()
print("mean spikes per 5 ms bin:", d["train_in"].mean().round(4), "(held-in),",
      d["train_out"].mean().round(4), "(held-out)")

## 3b. Smoothing
Raw spikes are 0/1/2 counts per 5 ms bin: too noisy to use directly. The baseline smooths each
neuron over time with a 40 ms Gaussian to estimate a firing rate.

In [ ]:
sd = 40 / 5   # 40 ms in 5 ms bins
n = 5         # neuron index
raw = d["train_in"][trial, :, n]
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(np.arange(140) * 5, raw, width=5, color="0.75", label="raw spike counts")
for s_ms, c in [(20, "tab:green"), (40, "tab:red"), (80, "tab:blue")]:
    ax.plot(np.arange(140) * 5, smooth(d["train_in"], s_ms / 5)[trial, :, n], c=c,
            label=f"Gaussian SD {s_ms} ms" + ("  (baseline)" if s_ms == 40 else ""))
ax.set(xlabel="time (ms)", ylabel="spikes / bin", title=f"Neuron {n}, trial {trial}")
ax.legend(frameon=False); plt.show()

## 3c. Fit the Poisson GLM baseline
For each held-out neuron, fit a Poisson regression from the log of the smoothed held-in rates
to that neuron's spike counts (training trials only), then predict the validation trials.

In [ ]:
from sklearn.linear_model import PoissonRegressor
tr = np.log(smooth(d["train_in"], sd).reshape(-1, 137) + 1e-4)
va_s = smooth(d["val_in"], sd)
va = np.log(va_s.reshape(-1, 137) + 1e-4)
ytr = d["train_out"].reshape(-1, 45)
pred = np.zeros((*va_s.shape[:2], 45))
t0 = time.time()
for c in range(45):
    m = PoissonRegressor(alpha=0.1, max_iter=500).fit(tr, ytr[:, c])
    pred[:, :, c] = m.predict(va).reshape(va_s.shape[:2])
print(f"fit 45 GLMs in {time.time()-t0:.0f}s")

## 3d. Predicted vs actual for held-out neurons (validation trials)

In [ ]:
null = mean_rate(d)
fig, axes = plt.subplots(2, 3, figsize=(13, 5), sharex=True)
for ax, c in zip(axes.flat, [0, 7, 15, 22, 30, 41]):
    ax.plot(np.arange(140) * 5, d["val_out"][:, :, c].mean(0), c="k", lw=2, label="actual (trial avg)")
    ax.plot(np.arange(140) * 5, pred[:, :, c].mean(0), c="tab:red", label="GLM prediction")
    ax.plot(np.arange(140) * 5, null[:, :, c].mean(0), c="0.6", ls="--", label="mean-rate null")
    ax.set_title(f"held-out neuron {c}", fontsize=10)
axes[0, 0].legend(frameon=False, fontsize=8)
for a in axes[1]: a.set_xlabel("time (ms)")
plt.tight_layout(); plt.show()

## 3e. Score it: co-bps with a confidence interval
The CI comes from resampling validation trials with replacement 200 times.

In [ ]:
score = co_bps(pred, d["val_out"], d["train_out"])
lo, hi = bootstrap_ci(pred, d["val_out"], d["train_out"])
print(f"smoothing + Poisson GLM  co-bps = {score:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

# per-neuron breakdown
per = []
for c in range(45):
    sl = (slice(None), slice(None), slice(c, c + 1))
    per.append(co_bps(pred[sl], d["val_out"][sl], d["train_out"][:, :, c:c+1]))
per = np.array(per)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].bar(range(45), np.sort(per)[::-1], color=np.where(np.sort(per)[::-1] > 0, "tab:blue", "tab:red"))
ax[0].axhline(0, c="k", lw=.8); ax[0].set(xlabel="held-out neuron (sorted)", ylabel="co-bps",
                                         title="Per-neuron co-bps")
ax[1].barh(["mean-rate null", "GLM baseline"], [0, score], xerr=[[0, score - lo], [0, hi - score]],
           color=["0.7", "tab:red"]); ax[1].set(xlabel="val co-bps", title="Overall")
plt.tight_layout(); plt.show()
print(f"{(per < 0).sum()} of 45 neurons are predicted worse than the null")

**Cross-check against the paper:** NLB'21 (Pei et al., Table 1) lists 0.211 for smoothing on MC_Maze. This notebook uses a 40 ms kernel, alpha 0.1 and the stricter training-null metric, so a lower number is expected here. The NLB reference settings for this dataset are 50 ms and alpha 0.01; `notebooks/step5_final_evaluation.ipynb` reproduces 0.212 with those settings and the official metric. Which split the paper's table uses is unconfirmed, so treat the comparison as approximate.

# Step 4: hyperparameter sweep on the internal dev fold
Hold out 15% of the *training* trials ("dev fold"), train on the rest, and compare settings by
dev co-bps. The official val split is not used to choose anything. Each run is ~1.5 min on the
GPU, so this section takes ~5 min.

In [ ]:
import argparse, torch
from neuralfm.train import parser, run

configs = {
    "dropout 0.2":                  dict(dropout=0.2),
    "dropout 0.5":                  dict(dropout=0.5),
    "dropout 0.5, wd 0.1":          dict(dropout=0.5, wd=0.1),
    "dropout 0.5, mask 0.5":        dict(dropout=0.5, mask_ratio=0.5),
}
results = {}
for i, (name, kw) in enumerate(configs.items()):
    a = parser().parse_args(["--run-name", f"nb_cfg{i}", "--epochs", "400", "--patience", "100",
                             "--no-mlflow", "--out", "../results/nb_runs"])
    for k, v in kw.items(): setattr(a, k, v)
    print("==", name)
    results[name] = run(a)

In [ ]:
names = list(results)
dev = [results[n]["dev_co_bps"] for n in names]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(names, dev, color=["tab:blue" if v < max(dev) else "tab:green" for v in dev])
ax.axvline(score, c="tab:red", ls="--", label=f"GLM baseline (val) {score:.3f}")
for i, v in enumerate(dev): ax.text(v + .003, i, f"{v:.3f}", va="center")
ax.set(xlabel="dev-fold co-bps", title="Sweep: choose by dev, not by val"); ax.legend(frameon=False)
ax.invert_yaxis(); plt.tight_layout(); plt.show()
print("Caution: the dev fold is ~258 trials and each config ran once (one seed), so treat gaps of ~0.01 as unconfirmed until repeated over several seeds.")

## Optional: how well does the winning model predict?
Reload the best checkpoint from the sweep and plot one held-out neuron against the GLM.

In [ ]:
from neuralfm.model import MaskedSpikeTransformer, ModelConfig
from neuralfm.train import predict
best = max(results, key=lambda n: results[n]["dev_co_bps"])   # chosen by dev score
ck = torch.load(f"../results/nb_runs/nb_cfg{names.index(best)}_s0.pt")
print("best config by dev:", best)
m = MaskedSpikeTransformer(ModelConfig(**ck["cfg"])).cuda(); m.load_state_dict(ck["state"])
tf = predict(m, d["val_in"], "cuda")[..., 137:]
print(f"transformer val co-bps = "
      f"{co_bps(tf, d['val_out'], d['train_out']):.3f}")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for a, c in zip(ax, [0, 15, 30]):
    t = np.arange(140) * 5
    a.plot(t, d["val_out"][:, :, c].mean(0), "k", lw=2, label="actual")
    a.plot(t, pred[:, :, c].mean(0), "tab:red", label="GLM")
    a.plot(t, tf[:, :, c].mean(0), "tab:green", label="transformer")
    a.set_title(f"held-out neuron {c}"); a.set_xlabel("time (ms)")
ax[0].legend(frameon=False); plt.tight_layout(); plt.show()